In [ ]:
"""
Preprocess MULTI dataset using h5ad file with pre-computed PCA.

Uses pre-computed X_pca from the h5ad file:
- multi100: X_pca[:, :100] - NO StandardScaler
- multi5:   X_pca[:, :5]   + StandardScaler
"""

import os
from typing import Dict, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from sklearn.preprocessing import StandardScaler

## Configuration

In [ ]:
# ===== CONFIGURATION =====
H5AD_PATH = "/path/to/op_train_multi_targets_0.h5ad"
OUTPUT_DIR = "/path/to/output_dir"

# Preprocessing parameters
PCA_DIMS = [5, 100]  # Dimensions to save
TRAIN_RATIO = 0.85  # 85% train, 15% test
SEED = 42

# Whitening options: create both versions for comparison
# - "none": no StandardScaler
# - "all": StandardScaler on all dims
# - "dim5_only": StandardScaler only on dim5
WHITENING_MODES = [
    "none",
    "all",
]  # Will create dim5/, dim100/, dim5_whitened/, dim100_whitened/

## 1. Load Pre-computed PCA from h5ad File

In [ ]:
# Load h5ad file with pre-computed PCA
print(f"Loading h5ad file: {H5AD_PATH}")
adata = sc.read_h5ad(H5AD_PATH)

print(f"\n=== AnnData Structure (before filtering) ===")
print(f"Shape: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"obsm keys: {list(adata.obsm.keys())}")

print(f"Number of donors: {adata.obs['donor'].nunique()}")
assert adata.obs['donor'].nunique() == 1, "Expected only one donor in the dataset."

# Extract pre-computed PCA and day labels
X_pca = adata.obsm["X_pca"]
day = adata.obs["day"].values

print(f"\n=== X_pca (pre-computed by scanpy) ===")
print(f"Shape: {X_pca.shape}")
print(f"Mean: {np.mean(X_pca):.6f}")
print(f"Std: {np.std(X_pca):.6f}")
print(f"Min: {np.min(X_pca):.4f}, Max: {np.max(X_pca):.4f}")

print(f"\n=== Day labels ===")
unique_days = sorted(np.unique(day))
print(f"Unique days: {unique_days}")
for d in unique_days:
    count = np.sum(day == d)
    print(f"  Day {d}: {count} cells")

## 2. Visualize PCA Statistics

In [ ]:
# Per-PC statistics
print("=== Per-PC Statistics ===")
print(f"{'PC':>4} {'Mean':>10} {'Std':>10}")
print("-" * 26)
for i in range(min(10, X_pca.shape[1])):
    print(f"{i + 1:>4} {np.mean(X_pca[:, i]):>10.4f} {np.std(X_pca[:, i]):>10.4f}")
print("...")

# Visualize variance per PC
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Variance per PC
pc_std = np.std(X_pca, axis=0)
axes[0].bar(range(1, len(pc_std) + 1), pc_std, alpha=0.7)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Standard Deviation")
axes[0].set_title("Std per PC (pre-computed X_pca)")
for dim in PCA_DIMS:
    axes[0].axvline(x=dim, color="r", linestyle="--", alpha=0.7, label=f"dim={dim}")
axes[0].legend()

# Distribution of all values
sample = X_pca.flatten()[::100]
axes[1].hist(sample, bins=100, alpha=0.7, edgecolor="black")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")
axes[1].set_title(
    f"Distribution of X_pca values (mean={np.mean(X_pca):.2f}, std={np.std(X_pca):.2f})"
)
axes[1].axvline(x=0, color="r", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 3. Visualize Per-Day Distributions

In [ ]:
# Per-day statistics
print("=== Per-Day Statistics (100-dim) ===")
print(f"{'Day':>4} {'Count':>8} {'Mean':>10} {'Std':>10}")
print("-" * 34)
for d in unique_days:
    mask = day == d
    X_d = X_pca[mask]
    print(f"{d:>4} {mask.sum():>8} {np.mean(X_d):>10.4f} {np.std(X_d):>10.4f}")

# Visualize PC1 vs PC2 colored by day
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All data colored by day
colors = plt.cm.viridis(np.linspace(0, 1, len(unique_days)))
for i, d in enumerate(unique_days):
    mask = day == d
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1], c=[colors[i]], alpha=0.3, s=1, label=f"Day {d}"
    )
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("PCA Projection (PC1 vs PC2) - Colored by Day")
axes[0].legend(markerscale=10)

# PC3 vs PC4
for i, d in enumerate(unique_days):
    mask = day == d
    axes[1].scatter(
        X_pca[mask, 2], X_pca[mask, 3], c=[colors[i]], alpha=0.3, s=1, label=f"Day {d}"
    )
axes[1].set_xlabel("PC3")
axes[1].set_ylabel("PC4")
axes[1].set_title("PCA Projection (PC3 vs PC4) - Colored by Day")
axes[1].legend(markerscale=10)

plt.tight_layout()
plt.show()

## 4. Split by Day and Create Train/Test Sets

In [ ]:
def split_by_day(
    X_pca: np.ndarray, day: np.ndarray, dim: int
) -> Tuple[Dict[int, np.ndarray], list]:
    """
    Split data by day, taking first `dim` PCs.

    Returns:
        day_data: Dictionary mapping day_index (0, 1, 2, 3) to data array
        unique_days: List of original day values [2, 3, 4, 7]
    """
    unique_days = sorted(np.unique(day))  # [2, 3, 4, 7]
    day_to_idx = {d: i for i, d in enumerate(unique_days)}

    print(f"Splitting into {len(unique_days)} time points (dim={dim})...")

    day_data = {}
    for d in unique_days:
        mask = day == d
        idx = day_to_idx[d]
        day_data[idx] = X_pca[mask, :dim]
        print(
            f"  Day {d} -> idx {idx}: {mask.sum()} samples, shape={day_data[idx].shape}"
        )

    return day_data, unique_days


def train_test_split(
    day_data: Dict[int, np.ndarray], train_ratio: float, seed: int
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray]]:
    """
    Split each day into train/test sets.
    """
    np.random.seed(seed)

    train_data = {}
    test_data = {}

    print(f"\nTrain/test split (ratio={train_ratio}):")
    for idx, data in day_data.items():
        n = len(data)
        n_train = int(n * train_ratio)

        indices = np.random.permutation(n)
        train_idx = indices[:n_train]
        test_idx = indices[n_train:]

        train_data[idx] = data[train_idx]
        test_data[idx] = data[test_idx]

        print(f"  idx {idx}: {len(train_idx)} train, {len(test_idx)} test")

    return train_data, test_data

## 5. Save Preprocessed Data

In [ ]:
def save_preprocessed(
    output_dir: str,
    dim: int,
    train_data: Dict[int, np.ndarray],
    test_data: Dict[int, np.ndarray],
    unique_days: list,
    whiten: bool = False,
):
    """
    Save preprocessed data.

    Args:
        whiten: If True, apply StandardScaler to PCA output.
                If False, save raw PCA values.
    """
    # Create directory name based on whitening
    if whiten:
        dim_dir = os.path.join(output_dir, f"dim{dim}_whitened")
    else:
        dim_dir = os.path.join(output_dir, f"dim{dim}")

    os.makedirs(dim_dir, exist_ok=True)
    print(f"\nSaving to {dim_dir}/")

    # Apply StandardScaler if whitening is enabled
    if whiten:
        all_train = np.concatenate(list(train_data.values()), axis=0)
        scaler = StandardScaler()
        scaler.fit(all_train)

        print(f"  [whiten=True] Applying StandardScaler")
        print(f"    mean: {scaler.mean_[:3]}...")  # Show first 3
        print(f"    std:  {scaler.scale_[:3]}...")

        # Transform both train and test
        for idx in train_data:
            train_data[idx] = scaler.transform(train_data[idx]).astype(np.float32)
            test_data[idx] = scaler.transform(test_data[idx]).astype(np.float32)

        # Verify
        all_scaled = np.concatenate(list(train_data.values()), axis=0)
        print(
            f"    After scaling: mean={np.mean(all_scaled):.6f}, std={np.std(all_scaled):.6f}"
        )

        # Save scaler
        scaler_path = os.path.join(dim_dir, "scaler.pkl")
        joblib.dump(scaler, scaler_path)
        print("  ✓ scaler.pkl")
    else:
        print(f"  [whiten=False] NO StandardScaler")
        scaler = None

        # Just convert to float32
        for idx in train_data:
            train_data[idx] = train_data[idx].astype(np.float32)
            test_data[idx] = test_data[idx].astype(np.float32)

        # Print stats
        all_data = np.concatenate(list(train_data.values()), axis=0)
        print(f"    Stats: mean={np.mean(all_data):.6f}, std={np.std(all_data):.6f}")

    # Save per-day data
    for idx in sorted(train_data.keys()):
        # Training data
        train_path = os.path.join(dim_dir, f"time_{idx}.npz")
        np.savez(train_path, pcs=train_data[idx])
        print(f"  ✓ time_{idx}.npz (train: {train_data[idx].shape})")

        # Test data
        test_dir = os.path.join(dim_dir, "test")
        os.makedirs(test_dir, exist_ok=True)
        test_path = os.path.join(test_dir, f"time_{idx}.npz")
        np.savez(test_path, pcs=test_data[idx])

    # Save metadata
    meta_path = os.path.join(dim_dir, "metadata.txt")
    with open(meta_path, "w") as f:
        f.write(f"Source: {H5AD_PATH}\n")
        f.write(f"PCA dimension: {dim}\n")
        f.write(f"StandardScaler (whitening): {whiten}\n")
        f.write(f"Train ratio: {TRAIN_RATIO}\n")
        f.write("Preprocessing: pre-computed X_pca from h5ad\n")
        f.write("\nPer-day breakdown:\n")
        for idx, d in enumerate(unique_days):
            f.write(
                f"  idx {idx} (day {d}): {len(train_data[idx])} train, {len(test_data[idx])} test\n"
            )
    print("  ✓ metadata.txt")

## 6. Process and Save All Dimensions

In [ ]:
# Process each dimension with both whitening options
for whiten_mode in WHITENING_MODES:
    whiten = whiten_mode == "all"

    print(f"\n{'#' * 60}")
    print(f"# WHITENING MODE: {whiten_mode} (whiten={whiten})")
    print(f"{'#' * 60}")

    for dim in PCA_DIMS:
        print(f"\n{'=' * 60}")
        print(f"Processing dim={dim}, whiten={whiten}")
        print(f"{'=' * 60}")

        # Split by day (need fresh copy each time since we modify in-place)
        day_data, unique_days_list = split_by_day(X_pca, day, dim)

        # Train/test split
        train_data, test_data = train_test_split(day_data, TRAIN_RATIO, SEED)

        # Save with appropriate whitening setting
        save_preprocessed(
            OUTPUT_DIR, dim, train_data, test_data, unique_days_list, whiten=whiten
        )

print(f"\n{'#' * 60}")
print(f"✓ Done! Data saved to: {OUTPUT_DIR}")
print(f"  - dim5/, dim100/           (no whitening)")
print(f"  - dim5_whitened/, dim100_whitened/  (with whitening)")
print(f"{'#' * 60}")

## 7. Verify Saved Data

In [ ]:
# Verify saved data for all versions
print("Verifying saved data...\n")

# All directories to check
dirs_to_check = ["dim5", "dim100", "dim5_whitened", "dim100_whitened"]

for dir_name in dirs_to_check:
    dim_dir = os.path.join(OUTPUT_DIR, dir_name)
    if not os.path.exists(dim_dir):
        print(f"=== {dir_name} === (not found, skipping)")
        continue

    print(f"=== {dir_name} ===")

    all_train = []
    for t in range(4):
        train_path = os.path.join(dim_dir, f"time_{t}.npz")
        test_path = os.path.join(dim_dir, "test", f"time_{t}.npz")

        train = np.load(train_path)["pcs"]
        test = np.load(test_path)["pcs"]
        all_train.append(train)

        print(
            f"  time_{t}: train={train.shape}, test={test.shape}, "
            f"mean={np.mean(train):.4f}, std={np.std(train):.4f}"
        )

    concat = np.concatenate(all_train, axis=0)
    print(
        f"  TOTAL: shape={concat.shape}, mean={np.mean(concat):.6f}, std={np.std(concat):.6f}"
    )
    print()